# Mission_Analysis_Test_File
# Generalized pipeline for cubesat post-deployment identification analysis

## Checking the discriminatory strength of BSTAR as a discriminator for the identification and elimination of candidate satellites. 

# Imports 

In [2]:
import numpy as np
import pandas as pd
import json
import os
from datetime import datetime, timezone
from astropy.time import Time
from astropy.coordinates import TEME, ITRS, EarthLocation, CartesianRepresentation
import astropy.units as u
from sgp4.api import Satrec, SatrecArray

# Constants

In [3]:
ST_BASE  = "https://www.space-track.org"
ST_LOGIN = f"{ST_BASE}/ajaxauth/login"
ST_QUERY = f"{ST_BASE}/basicspacedata/query"

CACHE_DIR = os.path.expanduser("~/mission_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# Authentication

In [4]:
def login(username, password, session):
    """Login to Space-Track and return session."""
    resp = session.post(ST_LOGIN, data={"identity": "nathandarby2022@gmail.com", "password": "Boobear32*Boobear32"})
    if resp.status_code == 200:
        print(f"Logged in to Space-Track successfully!")
    else:
        print(f"Login failed: {resp.status_code}")
    return session

# TLE Fetching

In [5]:
def fetch_norad_ids(intldes, session, force=False):
    """Fetch NORAD IDs for a launch group from satcat. Cached permanently."""
    cache_file = os.path.join(CACHE_DIR, f"{intldes}_satcat.json")
    
    if os.path.exists(cache_file) and not force:
        with open(cache_file) as f:
            cache = json.load(f)
        print(f"SATCAT loaded from cache — {len(cache['norad_ids'])} objects")
        return cache['norad_ids']
    
    print(f"Querying satcat for {intldes}...")
    url = (f"{ST_QUERY}/class/satcat"
           f"/INTLDES/{intldes}~~"
           f"/OBJECT_TYPE/Payload"
           f"/orderby/NORAD_CAT_ID"
           f"/format/json")
    
    resp = session.get(url)
    if resp.status_code != 200:
        print(f"satcat query failed: {resp.status_code}")
        return []
    
    objects   = resp.json()
    norad_ids = [obj['NORAD_CAT_ID'] for obj in objects]
    
    cache = {
        'intldes':    intldes,
        'fetched_at': datetime.now(timezone.utc).isoformat(),
        'norad_ids':  norad_ids,
        'objects':    [{'NORAD_CAT_ID': obj['NORAD_CAT_ID'],
                        'SATNAME':      obj['SATNAME']} for obj in objects]
    }
    with open(cache_file, 'w') as f:
        json.dump(cache, f, indent=2)
    
    print(f"Fetched {len(norad_ids)} payloads for {intldes}")
    return norad_ids


def fetch_current_tles(intldes, norad_ids, session, chunk_size=50, force=False):
    """Fetch current TLEs from gp class. Cached for 1 hour."""
    cache_file = os.path.join(CACHE_DIR, f"{intldes}_tles.json")
    
    if os.path.exists(cache_file) and not force:
        with open(cache_file) as f:
            cache = json.load(f)
        fetched_at = datetime.fromisoformat(cache['fetched_at'])
        age = datetime.now(timezone.utc) - fetched_at
        if age.total_seconds() < 3600:
            print(f"TLEs loaded from cache — {len(cache['tles'])} objects")
            return cache['tles']
        print("TLE cache expired — fetching fresh")
    
    all_tles = []
    for i in range(0, len(norad_ids), chunk_size):
        chunk   = norad_ids[i:i+chunk_size]
        ids_str = ','.join(str(n) for n in chunk)
        url = (f"{ST_QUERY}/class/gp"
               f"/NORAD_CAT_ID/{ids_str}"
               f"/decay_date/null-val"
               f"/orderby/NORAD_CAT_ID"
               f"/format/json")
        resp = session.get(url)
        print(f"   Chunk {i//chunk_size + 1}: status {resp.status_code}")
        if resp.status_code != 200 or not resp.text.strip():
            continue
        for obj in resp.json():
            if obj.get('TLE_LINE1') and obj.get('TLE_LINE2'):
                all_tles.append({
                    'name':  obj['OBJECT_NAME'],
                    'line1': obj['TLE_LINE1'],
                    'line2': obj['TLE_LINE2']
                })
    
    cache = {'fetched_at': datetime.now(timezone.utc).isoformat(), 'tles': all_tles}
    with open(cache_file, 'w') as f:
        json.dump(cache, f, indent=2)
    
    print(f"Fetched {len(all_tles)} TLEs for {intldes}")
    return all_tles


def fetch_historical_tles(intldes, norad_ids, launch_date, session,
                           days=60, chunk_size=50, force=False):
    """Fetch historical TLEs from gp_history. Cached permanently."""
    cache_file = os.path.join(CACHE_DIR, f"{intldes}_history.json")
    
    if os.path.exists(cache_file) and not force:
        with open(cache_file) as f:
            cache = json.load(f)
        print(f"Historical TLEs loaded from cache — {len(cache['tles'])} records")
        return cache['tles']
    
    # Parse launch date
    launch_dt  = pd.Timestamp(launch_date)
    end_date   = (launch_dt + pd.Timedelta(days=days)).strftime('%Y-%m-%d')
    start_date = launch_dt.strftime('%Y-%m-%d')
    
    print(f"Querying gp_history: {start_date} → {end_date}")
    print(f"WARNING: One-time query — caching permanently")
    
    all_tles = []
    for i in range(0, len(norad_ids), chunk_size):
        chunk   = norad_ids[i:i+chunk_size]
        ids_str = ','.join(str(n) for n in chunk)
        url = (f"{ST_QUERY}/class/gp_history"
               f"/NORAD_CAT_ID/{ids_str}"
               f"/epoch/{start_date}--{end_date}"
               f"/orderby/NORAD_CAT_ID,EPOCH"
               f"/format/json")
        resp = session.get(url)
        print(f"   Chunk {i//chunk_size + 1}: status {resp.status_code} "
              f"— {len(resp.json()) if resp.status_code == 200 else 0} records")
        if resp.status_code != 200 or not resp.text.strip():
            continue
        for obj in resp.json():
            if obj.get('TLE_LINE1') and obj.get('TLE_LINE2'):
                all_tles.append({
                    'name':  obj['OBJECT_NAME'],
                    'norad': obj['NORAD_CAT_ID'],
                    'epoch': obj['EPOCH'],
                    'line1': obj['TLE_LINE1'],
                    'line2': obj['TLE_LINE2']
                })
    
    cache = {
        'intldes':    intldes,
        'start_date': start_date,
        'end_date':   end_date,
        'fetched_at': datetime.now(timezone.utc).isoformat(),
        'tles':       all_tles
    }
    with open(cache_file, 'w') as f:
        json.dump(cache, f, indent=2)
    
    print(f"Cached {len(all_tles)} historical records")
    return all_tles


In [6]:
def compute_confirmation_timeline(historical_tles, launch_date):
    """
    Compute time to catalog and time to confirmation for each satellite.
    Returns dataframe with cataloging and confirmation timeline.
    """
    hist_df = pd.DataFrame(historical_tles)
    hist_df['epoch'] = pd.to_datetime(hist_df['epoch'], utc=True)
    
    # Confirmed = not TBA or OBJECT
    def is_confirmed(name):
        return not (name.startswith('TBA') or name.startswith('OBJECT'))
    
    hist_df['is_confirmed'] = hist_df['name'].apply(is_confirmed)
    
    launch_ts = pd.Timestamp(launch_date, tz='UTC')
    
    # First cataloged per NORAD
    first_cataloged = (hist_df.groupby('norad')['epoch']
                              .min()
                              .reset_index()
                              .rename(columns={'epoch': 'first_cataloged'}))
    
    # First confirmed per NORAD
    first_confirmed = (hist_df[hist_df['is_confirmed']]
                              .groupby('norad')['epoch']
                              .min()
                              .reset_index()
                              .rename(columns={'epoch': 'first_confirmed'}))
    
    # Unconfirmed name
    tba_names = (hist_df[~hist_df['is_confirmed']]
                        .groupby('norad')['name']
                        .first()
                        .reset_index()
                        .rename(columns={'name': 'unconfirmed_name'}))
    
    # Confirmed name
    confirmed_names = (hist_df[hist_df['is_confirmed']]
                              .groupby('norad')['name']
                              .first()
                              .reset_index()
                              .rename(columns={'name': 'confirmed_name'}))
    
    # Merge all
    df = (first_cataloged
          .merge(first_confirmed,  on='norad', how='left')
          .merge(tba_names,        on='norad', how='left')
          .merge(confirmed_names,  on='norad', how='left'))
    
    df['days_to_catalog']   = (df['first_cataloged'] - launch_ts).dt.total_seconds() / 86400
    df['days_to_confirm']   = (df['first_confirmed'] - launch_ts).dt.total_seconds() / 86400
    df['confirmation_lag']  = df['days_to_confirm'] - df['days_to_catalog']
    df = df.sort_values('days_to_confirm').reset_index(drop=True)
    
    # Summary
    print(f"Confirmation Timeline — {launch_date}")
    print(f"   Total objects:         {len(df)}")
    print(f"   Mean days to catalog:  {df['days_to_catalog'].mean():.1f}")
    print(f"   Mean days to confirm:  {df['days_to_confirm'].mean():.1f}")
    print(f"   Mean confirmation lag: {df['confirmation_lag'].mean():.1f}")
    print(f"   Confirmed:             {df['confirmed_name'].notna().sum()}")
    print(f"   Unconfirmed:           {df['confirmed_name'].isna().sum()}")
    
    return df

# Propagation

In [7]:
def propagate_tles(tles, duration_hrs=24, num_steps=2000):
    """Vectorized SGP4 propagation for all TLEs."""
    now    = Time(datetime.now(timezone.utc))
    times  = now + np.linspace(0, duration_hrs * 3600, num_steps) * u.s
    
    sats   = SatrecArray([Satrec.twoline2rv(t['line1'], t['line2']) for t in tles])
    jd1    = np.array([t.jd1 for t in times])
    jd2    = np.array([t.jd2 for t in times])
    
    e, r_teme, v_teme = sats.sgp4(jd1, jd2)
    
    # Convert to lat/lon/alt
    lats = np.zeros((len(tles), num_steps))
    lons = np.zeros((len(tles), num_steps))
    alts = np.zeros((len(tles), num_steps))
    
    for i in range(len(tles)):
        for j in range(num_steps):
            if e[i, j] != 0:
                continue
            r     = CartesianRepresentation(r_teme[i, j] * u.km)
            teme  = TEME(r, obstime=times[j])
            itrs  = teme.transform_to(ITRS(obstime=times[j]))
            loc   = EarthLocation(*itrs.cartesian.xyz)
            lats[i, j] = loc.lat.deg
            lons[i, j] = loc.lon.deg
            alts[i, j] = loc.height.to(u.km).value
    
    print(f"Propagation complete — {len(tles)} satellites, {num_steps} steps")
    print(f"   Errors: {(e != 0).sum()} non-zero error codes")
    return e, r_teme, v_teme, lats, lons, alts, times

# TLE Dataframe

In [8]:
def parse_bstar(s):
    """Parse compressed scientific notation BSTAR from TLE."""
    s = s.strip()
    for i in range(1, len(s)):
        if s[i] in '+-':
            mantissa = float('0.' + s[:i])
            exp      = int(s[i:])
            return mantissa * (10 ** exp)
    return float(s)

def build_tle_dataframe(tles, alts, e):
    """Build analytical dataframe from TLEs and propagated data."""
    if len(tles) == 0:
        print("No TLEs to process")
        return pd.DataFrame()
    
    rows = []
    for i, t in enumerate(tles):
        line1 = t['line1']
        line2 = t['line2']
        
        intldes_raw = line1[9:17].strip()
        year        = intldes_raw[:2]
        num         = intldes_raw[2:5]
        piece       = intldes_raw[5:]
        full_year   = f"19{year}" if int(year) >= 57 else f"20{year}"
        intldes     = f"{full_year}-{num}{piece}"
        
        valid = alts[i][e[i] == 0]
        
        rows.append({
            'idx':          i,
            'name':         t['name'],
            'intldes':      intldes,
            'norad':        line1[2:7].strip(),
            'bstar':        parse_bstar(line1[53:61]),
            'mean_motion':  float(line2[52:63].strip()),
            'inclination':  float(line2[8:16].strip()),
            'eccentricity': float('0.' + line2[26:33].strip()),
            'raan':         float(line2[17:25].strip()),
            'arg_perigee':  float(line2[34:42].strip()),
            'mean_anomaly': float(line2[43:51].strip()),
            'alt_mean':     valid.mean() if len(valid) > 0 else np.nan,
            'alt_min':      valid.min()  if len(valid) > 0 else np.nan,
            'alt_max':      valid.max()  if len(valid) > 0 else np.nan,
            'period_min':   1440 / float(line2[52:63].strip())
        })
    
    df = pd.DataFrame(rows)
    df['alt_range']        = df['alt_max'] - df['alt_min']
    df['decay_rate_km_day'] = [
        np.polyfit(np.linspace(0, 24, alts.shape[1])[e[i] == 0],
                   alts[i][e[i] == 0], 1)[0] * 24
        if (e[i] == 0).sum() > 10 else np.nan
        for i in range(len(tles))
    ]
    return df

# Clustering 

In [40]:
def assign_cluster_dynamic(alt, alt_array):
    """
    Assign cluster based on dynamic percentile boundaries 
    computed from the actual altitude distribution.
    """
    p25 = np.percentile(alt_array, 25)
    p50 = np.percentile(alt_array, 50)
    p75 = np.percentile(alt_array, 75)
    
    if alt < p25:
        return f'A — Lower Quartile (<{p25:.0f}km)'
    elif alt < p50:
        return f'B — Lower Mid ({p25:.0f}-{p50:.0f}km)'
    elif alt < p75:
        return f'C — Upper Mid ({p50:.0f}-{p75:.0f}km)'
    else:
        return f'D — Upper Quartile (>{p75:.0f}km)'


def cluster_dynamic(alt_df):
    """Apply dynamic clustering to a dataframe with alt column."""
    alt_array = alt_df['alt'].values
    alt_df['cluster'] = alt_df['alt'].apply(
        lambda a: assign_cluster_dynamic(a, alt_array)
    )
    
    print(f"   Dynamic cluster boundaries:")
    print(f"   P25: {np.percentile(alt_array, 25):.0f}km")
    print(f"   P50: {np.percentile(alt_array, 50):.0f}km")
    print(f"   P75: {np.percentile(alt_array, 75):.0f}km")
    
    for cluster, group in alt_df.groupby('cluster'):
        print(f"   {cluster} — {len(group)} satellites "
              f"({group['alt'].min():.0f}-{group['alt'].max():.0f}km)")
    
    return alt_df

# BSTAR Correlation

In [73]:
def parse_bstar(s):
    s = s.strip()
    for i in range(1, len(s)):
        if s[i] in '+-':
            return float('0.' + s[:i]) * (10 ** int(s[i:]))
    return float(s)

def compute_bstar_correlation(df, exclude_maneuvering=True):
    """Compute BSTAR vs decay rate correlation."""
    df_clean = df.copy()
    
    if exclude_maneuvering:
        df_clean = df_clean[df_clean['alt_range'] < 50]
        print(f"Excluded {len(df) - len(df_clean)} maneuvering satellites")
    
    corr_raw = df_clean['bstar'].corr(df_clean['decay_rate_km_day'])
    r2       = corr_raw ** 2
    
    print(f"BSTAR vs decay rate correlation:")
    print(f"   Pearson r:  {corr_raw:.4f}")
    print(f"   R²:         {r2:.4f}")
    print(f"   Explained variance: {r2*100:.1f}%")
    
    return corr_raw, r2

# RTN Separation

In [11]:
def build_hist_dataframe(historical_tles, launch_date):
    """Build historical TLE dataframe."""
    hist_df = pd.DataFrame(historical_tles)
    hist_df['epoch'] = pd.to_datetime(hist_df['epoch'], utc=True)
    hist_df['date']  = hist_df['epoch'].dt.date
    hist_df['days_since_launch'] = (
        hist_df['epoch'] - pd.Timestamp(launch_date, tz='UTC')
    ).dt.total_seconds() / 86400
    return hist_df


def compute_rtn_separation(hist_df, launch_date, min_sats=5):
    """Compute RTN pairwise separation over time."""
    daily_tles  = (hist_df.sort_values('epoch')
                          .groupby(['norad', 'date'])
                          .first()
                          .reset_index())
    unique_days = sorted(daily_tles['date'].unique())
    
    rtn_results = []
    
    for day in unique_days:
        day_tles = daily_tles[daily_tles['date'] == day]
        day_tles = day_tles[day_tles['line1'].notna() & day_tles['line2'].notna()]
        
        if len(day_tles) < min_sats:
            continue
        
        eval_time = Time(f"{day}T12:00:00", scale='utc')
        jd1, jd2  = eval_time.jd1, eval_time.jd2
        
        positions  = []
        velocities = []
        
        for _, row in day_tles.iterrows():
            try:
                sat     = Satrec.twoline2rv(row['line1'], row['line2'])
                e, r, v = sat.sgp4(jd1, jd2)
                if e == 0:
                    positions.append(np.array(r))
                    velocities.append(np.array(v))
            except:
                continue
        
        if len(positions) < min_sats:
            continue
        
        positions  = np.array(positions)
        velocities = np.array(velocities)
        
        r_seps, t_seps, n_seps = [], [], []
        
        for a in range(len(positions)):
            R_hat = positions[a] / np.linalg.norm(positions[a])
            N_hat = np.cross(positions[a], velocities[a])
            N_hat = N_hat / np.linalg.norm(N_hat)
            T_hat = np.cross(N_hat, R_hat)
            T_hat = T_hat / np.linalg.norm(T_hat)
            
            for b in range(a+1, len(positions)):
                dr = positions[b] - positions[a]
                r_seps.append(np.abs(np.dot(dr, R_hat)))
                t_seps.append(np.abs(np.dot(dr, T_hat)))
                n_seps.append(np.abs(np.dot(dr, N_hat)))
        
        days_since = (pd.Timestamp(day) - pd.Timestamp(launch_date)).days
        
        rtn_results.append({
            'date':              day,
            'days_since_launch': days_since,
            'n_sats':            len(positions),
            'mean_R':            np.mean(r_seps),
            'mean_T':            np.mean(t_seps),
            'mean_N':            np.mean(n_seps),
            'median_R':          np.median(r_seps),
            'median_T':          np.median(t_seps),
            'median_N':          np.median(n_seps),
        })
    
    rtn_df = pd.DataFrame(rtn_results)
    rtn_df['rsi_R'] = rtn_df['mean_R'] / rtn_df['mean_R'].iloc[0]
    rtn_df['rsi_T'] = rtn_df['mean_T'] / rtn_df['mean_T'].iloc[0]
    rtn_df['rsi_N'] = rtn_df['mean_N'] / rtn_df['mean_N'].iloc[0]
    
    return rtn_df

# Optimal Window

In [53]:
def compute_optimal_window(rtn_df, min_sats_threshold=None):
    if min_sats_threshold:
        rtn_clean = rtn_df[rtn_df['n_sats'] >= min_sats_threshold].copy()
    else:
        rtn_clean = rtn_df.copy()
    
    if len(rtn_clean) < 3:
        print("Insufficient data for window calculation")
        return None, None, None, rtn_clean
    
    # Smooth mean_R before computing gradient
    rtn_clean['mean_R_smooth'] = rtn_clean['mean_R'].rolling(
        window=5, center=True, min_periods=1
    ).mean()
    
    rtn_clean['radial_growth_rate'] = np.gradient(
        rtn_clean['mean_R_smooth'], rtn_clean['days_since_launch']
    )
    
    # Only look for peak after day 10 to avoid early noise
    rtn_post10 = rtn_clean[rtn_clean['days_since_launch'] >= 10]
    
    if rtn_post10.empty:
        peak_idx = rtn_clean['radial_growth_rate'].idxmax()
    else:
        peak_idx = rtn_post10['radial_growth_rate'].idxmax()
    
    peak_day  = rtn_clean.loc[peak_idx, 'days_since_launch']
    peak_rate = rtn_clean.loc[peak_idx, 'radial_growth_rate']
    half_peak = peak_rate * 0.5
    
    # Only look for 50% decay after peak day
    after_peak = rtn_clean[rtn_clean['days_since_launch'] > peak_day]
    below_half = after_peak[after_peak['radial_growth_rate'] < half_peak]
    
    inflection_day = below_half.iloc[0]['days_since_launch'] if len(below_half) > 0 else None
    
    print(f"Optimal Identification Window:")
    print(f"   Peak growth rate:  {peak_rate:.2f} km/day at day {peak_day}")
    print(f"   50% decay point:   day {inflection_day}")
    print(f"   Optimal window:    days {rtn_clean['days_since_launch'].iloc[0]:.0f} — {inflection_day}")
    
    return peak_day, peak_rate, inflection_day, rtn_clean

# Full Pipeline

In [74]:
def run_mission_analysis(intldes, launch_date, session, history_days=60):
    
    print(f"\n{'='*60}")
    print(f"Mission Analysis: {intldes}")
    print(f"Launch Date:      {launch_date}")
    print(f"{'='*60}\n")
    
    # Step 1 — fetch NORAD IDs
    print("Step 1 — Fetching NORAD IDs...")
    norad_ids = fetch_norad_ids(intldes, session)
    print(f"   Total payloads cataloged: {len(norad_ids)}")
    
    # Step 2 — fetch historical TLEs
    print("\nStep 2 — Fetching historical TLEs...")
    historical_tles = fetch_historical_tles(
        intldes, norad_ids, launch_date, session, days=history_days
    )
    
    # Step 3 — build historical dataframe
    print("\nStep 3 — Building historical dataframe...")
    hist_df = build_hist_dataframe(historical_tles, launch_date)
    unique_sats = hist_df['norad'].nunique()
    unique_days = hist_df['date'].nunique()
    print(f"   Total records:       {len(hist_df)}")
    print(f"   Unique satellites:   {unique_sats}")
    print(f"   Days with data:      {unique_days}")
    print(f"   Epoch range:         {hist_df['epoch'].min().date()} → {hist_df['epoch'].max().date()}")
    
    # Step 4 — confirmation timeline
    print("\nStep 4 — Computing confirmation timeline...")
    confirm_df = compute_confirmation_timeline(historical_tles, launch_date)
    
    # Step 5 — cluster from earliest historical TLEs
    print("\nStep 5 — Clustering from earliest historical TLEs...")
    first_hist = (hist_df.sort_values('epoch')
                         .groupby('norad')
                         .first()
                         .reset_index())
    
    early_alts = []
    for _, row in first_hist.iterrows():
        try:
            sat     = Satrec.twoline2rv(row['line1'], row['line2'])
            t       = Time(row['epoch'])
            e, r, v = sat.sgp4(t.jd1, t.jd2)
            if e == 0:
                alt = np.linalg.norm(r) - 6371.0
                early_alts.append({'norad': row['norad'],
                                    'name':  row['name'],
                                    'alt':   alt})
        except:
            continue
    
    early_alt_df = pd.DataFrame(early_alts)
    alt_array    = early_alt_df['alt'].values
    
    Q1  = np.percentile(alt_array, 25)
    Q3  = np.percentile(alt_array, 75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    core     = early_alt_df[(early_alt_df['alt'] >= lower_bound) &
                             (early_alt_df['alt'] <= upper_bound)].copy()
    outliers = early_alt_df[(early_alt_df['alt'] < lower_bound) |
                             (early_alt_df['alt'] > upper_bound)].copy()
    
    print(f"   Total satellites:   {len(early_alt_df)}")
    print(f"   Core cluster:       {len(core)} satellites ({core['alt'].min():.0f}-{core['alt'].max():.0f}km)")
    print(f"   Outliers:           {len(outliers)} satellites")
    for _, row in outliers.iterrows():
        print(f"      {row['name']:<30} {row['alt']:.0f}km")
    
    core['cluster']     = 'Core'
    outliers['cluster'] = 'Outlier'
    early_alt_df        = pd.concat([core, outliers])
    cluster_norads      = core['norad'].tolist()
    hist_cluster        = hist_df[hist_df['norad'].isin(cluster_norads)]
    cluster_focus       = 'Core'
    
    print(f"\n   Focus cluster:      Core")
    print(f"   Satellites:         {len(cluster_norads)}")
    print(f"   Historical records: {len(hist_cluster)}")

    # Step 5.5 — BSTAR analysis
    print("\nStep 5.5 — BSTAR analysis...")

    bstar_rows = []
    for norad in cluster_norads:
        sat_hist = hist_df[hist_df['norad'] == norad].sort_values('epoch')
        if len(sat_hist) < 5:
            continue
        
        first = sat_hist.iloc[0]
        last  = sat_hist.iloc[-1]
        
        try:
            first_sat  = Satrec.twoline2rv(first['line1'], first['line2'])
            last_sat   = Satrec.twoline2rv(last['line1'],  last['line2'])
            t_first    = Time(first['epoch'])
            t_last     = Time(last['epoch'])
            
            ef, rf, vf = first_sat.sgp4(t_first.jd1, t_first.jd2)
            el, rl, vl = last_sat.sgp4(t_last.jd1,   t_last.jd2)
            
            if ef != 0 or el != 0:
                continue
            
            alt_first  = np.linalg.norm(rf) - 6371.0
            alt_last   = np.linalg.norm(rl) - 6371.0
            dt_days    = (t_last - t_first).to(u.day).value
            decay_rate = (alt_last - alt_first) / dt_days if dt_days > 0 else np.nan
            
            bstar_rows.append({
                'norad':       norad,
                'name':        first['name'],
                'bstar':       parse_bstar(last['line1'][53:61]),  # most recent BSTAR
                'alt_first':   alt_first,
                'alt_last':    alt_last,
                'total_decay': alt_last - alt_first,
                'decay_rate':  decay_rate,
                'dt_days':     dt_days
            })
        except Exception as ex:
            print(f"   Skipping {norad}: {ex}")
            continue

    if len(bstar_rows) == 0:
        print("   No valid BSTAR data computed")
        bstar_df   = pd.DataFrame()
        corr_daily = np.nan
        corr_total = np.nan
        r2_daily   = np.nan
        r2_total   = np.nan
    else:
        bstar_df    = pd.DataFrame(bstar_rows)
        bstar_clean = bstar_df[bstar_df['total_decay'] < 0].copy()
        
        corr_daily = bstar_clean['bstar'].corr(bstar_clean['decay_rate'])
        r2_daily   = corr_daily ** 2
        corr_total = bstar_clean['bstar'].corr(bstar_clean['total_decay'])
        r2_total   = corr_total ** 2
        
        print(f"   Satellites analyzed:         {len(bstar_df)}")
        print(f"   After removing outliers:     {len(bstar_clean)}")
        print(f"   BSTAR vs day-to-day decay:")
        print(f"      Pearson r:                {corr_daily:.4f}")
        print(f"      R²:                       {r2_daily:.4f}")
        print(f"      Explained variance:       {r2_daily*100:.1f}%")
        print(f"   BSTAR vs total 60-day decay:")
        print(f"      Pearson r:                {corr_total:.4f}")
        print(f"      R²:                       {r2_total:.4f}")
        print(f"      Explained variance:       {r2_total*100:.1f}%")
    # Step 6 — RTN separation
    print("\nStep 6 — Computing RTN separation...")
    rtn_df = compute_rtn_separation(hist_cluster, launch_date)
    
    if rtn_df.empty:
        print("Insufficient data for RTN analysis")
        return None
    
    # Step 7 — optimal window
    print("\nStep 7 — Computing optimal identification window...")
    min_thresh = max(5, int(len(cluster_norads) * 0.85))
    peak_day, peak_rate, inflection_day, rtn_clean = compute_optimal_window(
        rtn_df, min_sats_threshold=min_thresh
    )
    
    # Step 8 — plot
    print("\nStep 8 — Plotting RTN separation...")
    plot_rtn(rtn_df, intldes)
    
    print(f"\n{'='*60}")
    print(f"Summary: {intldes}")
    print(f"   Total payloads:        {len(norad_ids)}")
    print(f"   Unique sats in hist:   {unique_sats}")
    print(f"   Days with TLE data:    {unique_days}")
    print(f"   Mean days to catalog:  {confirm_df['days_to_catalog'].mean():.1f}")
    print(f"   Mean days to confirm:  {confirm_df['days_to_confirm'].mean():.1f}")
    print(f"   Confirmation lag:      {confirm_df['confirmation_lag'].mean():.1f}")
    print(f"   Focus cluster:         {cluster_focus}")
    print(f"   Cluster satellites:    {len(cluster_norads)}")
    print(f"   Optimal window:        days {rtn_df['days_since_launch'].iloc[0]:.0f} — {inflection_day}")
    print(f"   BSTAR daily corr:      {corr_daily:.4f} (R²={r2_daily:.3f})")
    print(f"   BSTAR total corr:      {corr_total:.4f} (R²={r2_total:.3f})")
    print(f"{'='*60}\n")
    
    return {
        'intldes':          intldes,
        'launch_date':      launch_date,
        'norad_ids':        norad_ids,
        'hist_df':          hist_df,
        'confirm_df':       confirm_df,
        'early_alt_df':     early_alt_df,
        'rtn_df':           rtn_df,
        'rtn_clean':        rtn_clean,
        'cluster_focus':    cluster_focus,
        'cluster_norads':   cluster_norads,
        'peak_day':         peak_day,
        'peak_rate':        peak_rate,
        'inflection_day':   inflection_day,
        'bstar_df':         bstar_df,
        'corr_daily':       corr_daily,
        'corr_total':       corr_total,
        'r2_daily':         r2_daily,
        'r2_total':    r2_total,
    }

# Running Mission Analysis 

In [75]:
import requests
session = requests.Session()
session.post(ST_LOGIN, data={"identity": "nathandarby2022@gmail.com", "password": "Boobear32*Boobear32"})

results_t7 = run_mission_analysis('2023-054', '2023-04-15', session,
                          history_days=60) 


Mission Analysis: 2023-054
Launch Date:      2023-04-15

Step 1 — Fetching NORAD IDs...
SATCAT loaded from cache — 48 objects
   Total payloads cataloged: 48

Step 2 — Fetching historical TLEs...
Historical TLEs loaded from cache — 6565 records

Step 3 — Building historical dataframe...
   Total records:       6565
   Unique satellites:   47
   Days with data:      58
   Epoch range:         2023-04-15 → 2023-06-13

Step 4 — Computing confirmation timeline...
Confirmation Timeline — 2023-04-15
   Total objects:         47
   Mean days to catalog:  8.9
   Mean days to confirm:  21.2
   Mean confirmation lag: 11.6
   Confirmed:             41
   Unconfirmed:           6

Step 5 — Clustering from earliest historical TLEs...
   Total satellites:   47
   Core cluster:       36 satellites (515-516km)
   Outliers:           11 satellites
      OBJECT A                       689km
      OBJECT AP                      695km
      OBJECT AQ                      695km
      OBJECT AR            


Summary: 2023-054
   Total payloads:        48
   Unique sats in hist:   47
   Days with TLE data:    58
   Mean days to catalog:  8.9
   Mean days to confirm:  21.2
   Confirmation lag:      11.6
   Focus cluster:         Core
   Cluster satellites:    36
   Optimal window:        days 3 — 22
   BSTAR daily corr:      -0.5820 (R²=0.339)
   BSTAR total corr:      -0.5805 (R²=0.337)



# Compute Prop Error

In [76]:
def compute_propagation_error(hist_df, cluster_norads, launch_date, min_obs=5):
    """
    For each satellite, propagate first TLE forward and compare 
    against actual TLE positions over 60 days.
    Returns per-satellite and fleet-wide error growth.
    """
    daily_tles = (hist_df[hist_df['norad'].isin(cluster_norads)]
                         .sort_values('epoch')
                         .groupby(['norad', 'date'])
                         .first()
                         .reset_index())
    
    all_errors  = []  # per satellite per day
    fleet_daily = {}  # day → list of errors across satellites
    
    for norad in cluster_norads:
        sat_hist = daily_tles[daily_tles['norad'] == norad].sort_values('date')
        
        if len(sat_hist) < min_obs:
            continue
        
        # First TLE — use as baseline propagator
        first_row = sat_hist.iloc[0]
        try:
            sat_t0    = Satrec.twoline2rv(first_row['line1'], first_row['line2'])
            t0        = Time(first_row['epoch'])
            days_t0   = (pd.Timestamp(first_row['date']) - 
                         pd.Timestamp(launch_date)).days
        except:
            continue
        
        # Propagate to each subsequent TLE epoch
        for _, row in sat_hist.iterrows():
            try:
                # Actual position from updated TLE
                sat_tn    = Satrec.twoline2rv(row['line1'], row['line2'])
                t_eval    = Time(row['epoch'])
                jd1, jd2  = t_eval.jd1, t_eval.jd2
                
                en, rn, vn = sat_tn.sgp4(jd1, jd2)  # actual position
                e0, r0, v0 = sat_t0.sgp4(jd1, jd2)  # propagated from t0
                
                if en != 0 or e0 != 0:
                    continue
                
                error_km   = np.linalg.norm(np.array(r0) - np.array(rn))
                days_since = (pd.Timestamp(row['date']) - 
                              pd.Timestamp(launch_date)).days
                dt_from_t0 = days_since - days_t0
                
                all_errors.append({
                    'norad':      norad,
                    'name':       first_row['name'],
                    'days_since': days_since,
                    'dt_from_t0': dt_from_t0,
                    'error_km':   error_km
                })
                
                if dt_from_t0 not in fleet_daily:
                    fleet_daily[dt_from_t0] = []
                fleet_daily[dt_from_t0].append(error_km)
                
            except:
                continue
    
    error_df = pd.DataFrame(all_errors)
    
    # Fleet-wide daily statistics
    fleet_rows = []
    for dt, errors in sorted(fleet_daily.items()):
        if dt < 0:
            continue
        fleet_rows.append({
            'dt_from_first_tle': dt,
            'mean_error':        np.mean(errors),
            'median_error':      np.median(errors),
            'p75_error':         np.percentile(errors, 75),
            'p90_error':         np.percentile(errors, 90),
            'n_sats':            len(errors)
        })
    
    fleet_df = pd.DataFrame(fleet_rows)
    
    return error_df, fleet_df


def plot_propagation_error(fleet_df, intldes, optimal_window_end=None):
    """Plot fleet-wide propagation error growth."""
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=fleet_df['dt_from_first_tle'], y=fleet_df['median_error'],
        mode='lines+markers', name='Median Error',
        line=dict(color='cyan', width=2)
    ))
    fig.add_trace(go.Scatter(
        x=fleet_df['dt_from_first_tle'], y=fleet_df['mean_error'],
        mode='lines+markers', name='Mean Error',
        line=dict(color='orange', width=2)
    ))
    fig.add_trace(go.Scatter(
        x=fleet_df['dt_from_first_tle'], y=fleet_df['p75_error'],
        mode='lines+markers', name='75th Percentile',
        line=dict(color='yellow', width=1, dash='dash')
    ))
    fig.add_trace(go.Scatter(
        x=fleet_df['dt_from_first_tle'], y=fleet_df['p90_error'],
        mode='lines+markers', name='90th Percentile',
        line=dict(color='red', width=1, dash='dash')
    ))
    
    # Mark optimal window end
    if optimal_window_end:
        fig.add_vline(
            x=optimal_window_end,
            line_dash='dash', line_color='lime',
            annotation_text=f'Optimal window end (day {optimal_window_end})',
            annotation_font_color='lime'
        )
    
    # Refresh zones
    fig.add_vrect(x0=0,  x1=2,  fillcolor='green',  opacity=0.1,
                  annotation_text='Refresh Zone',  annotation_position='top left')
    fig.add_vrect(x0=2,  x1=7,  fillcolor='yellow', opacity=0.1,
                  annotation_text='Caution Zone',  annotation_position='top left')
    fig.add_vrect(x0=7,  x1=fleet_df['dt_from_first_tle'].max(),
                  fillcolor='red', opacity=0.05,
                  annotation_text='Stale Zone', annotation_position='top left')
    
    fig.update_layout(
        title=f'TLE Propagation Error Growth — {intldes}',
        xaxis_title='Days Since First TLE',
        yaxis_title='Position Error (km)',
        paper_bgcolor='black', plot_bgcolor='black',
        font=dict(color='white'),
        xaxis=dict(gridcolor='gray'),
        yaxis=dict(gridcolor='gray'),
        legend=dict(bgcolor='black')
    )
    fig.show()
    
    print(f"\nPropagation Error Summary:")
    print(f"{'Days from t0':>14} {'Median (km)':>12} {'Mean (km)':>10} "
          f"{'P75 (km)':>10} {'P90 (km)':>10} {'N sats':>8}")
    print("-" * 68)
    for _, row in fleet_df.iterrows():
        print(f"{row['dt_from_first_tle']:>14.0f} {row['median_error']:>12.2f} "
              f"{row['mean_error']:>10.2f} {row['p75_error']:>10.2f} "
              f"{row['p90_error']:>10.2f} {row['n_sats']:>8.0f}")

In [77]:
error_df, fleet_df = compute_propagation_error(
    results_t7['hist_df'], 
    results_t7['cluster_norads'], 
    '2023-04-15'
)
plot_propagation_error(fleet_df, '2023-054', optimal_window_end=22)


Propagation Error Summary:
  Days from t0  Median (km)  Mean (km)   P75 (km)   P90 (km)   N sats
--------------------------------------------------------------------
             0         0.00       0.00       0.00       0.00       36
             1         0.20       0.84       0.84       2.61       29
             2         0.93       2.03       1.87       4.60       33
             3         1.16       3.69       3.51      10.85       36
             4         3.65       7.32       7.72      18.72       36
             5        10.56      18.15      19.70      37.46       36
             6        17.96      35.54      27.73      70.29       36
             7        21.52      63.77      45.26     107.89       36
             8        28.19     125.50      65.53     168.78       36
             9        37.39     177.92      76.89     208.49       36
            10        48.98     237.13      90.75     245.70       36
            11        60.76     285.93     121.64     312.57   

In [80]:
def analyze_per_satellite_error(error_df, intldes, top_n=10):
    """
    Identify best and worst behaved satellites by TLE error growth rate.
    """
    # Fit linear slope to error vs time per satellite
    sat_slopes = []
    for norad, group in error_df.groupby('norad'):
        group = group[group['dt_from_t0'] > 0].sort_values('dt_from_t0')
        if len(group) < 5:
            continue
        
        # Fit error vs days
        coeffs = np.polyfit(group['dt_from_t0'], group['error_km'], 1)
        slope  = coeffs[0]  # km per day error growth
        
        # Also compute error at day 7 and day 14
        err_7  = group[group['dt_from_t0'] <= 7]['error_km'].max() if len(group[group['dt_from_t0'] <= 7]) > 0 else np.nan
        err_14 = group[group['dt_from_t0'] <= 14]['error_km'].max() if len(group[group['dt_from_t0'] <= 14]) > 0 else np.nan
        
        sat_slopes.append({
            'norad':       norad,
            'name':        group['name'].iloc[0],
            'slope_km_day': slope,
            'error_day7':  err_7,
            'error_day14': err_14,
            'n_obs':       len(group)
        })
    
    slope_df = pd.DataFrame(sat_slopes).sort_values('slope_km_day')
    
    # Plot best and worst
    fig = go.Figure()
    
    # Best behaved
    best  = slope_df.head(top_n)
    worst = slope_df.tail(top_n)
    
    for _, row in best.iterrows():
        sat_data = error_df[error_df['norad'] == row['norad']].sort_values('dt_from_t0')
        fig.add_trace(go.Scatter(
            x=sat_data['dt_from_t0'],
            y=sat_data['error_km'],
            mode='lines',
            name=f"✓ {row['name']}",
            line=dict(color='cyan', width=1),
            opacity=0.7
        ))
    
    for _, row in worst.iterrows():
        sat_data = error_df[error_df['norad'] == row['norad']].sort_values('dt_from_t0')
        fig.add_trace(go.Scatter(
            x=sat_data['dt_from_t0'],
            y=sat_data['error_km'],
            mode='lines',
            name=f"✗ {row['name']}",
            line=dict(color='red', width=1),
            opacity=0.7
        ))
    
    fig.update_layout(
        title=f'Per-Satellite TLE Error Growth — {intldes}',
        xaxis_title='Days Since First TLE',
        yaxis_title='Position Error (km)',
        paper_bgcolor='black', plot_bgcolor='black',
        font=dict(color='white'),
        xaxis=dict(gridcolor='gray'),
        yaxis=dict(gridcolor='gray'),
        legend=dict(bgcolor='black', font=dict(size=9))
    )
    fig.show()
    
    print(f"\nBest behaved TLEs (slowest error growth):")
    print(f"{'Name':<30} {'Slope (km/day)':>15} {'Error day7':>12} {'Error day14':>12}")
    print("-" * 72)
    for _, row in best.iterrows():
        print(f"{row['name']:<30} {row['slope_km_day']:>15.2f} "
              f"{row['error_day7']:>12.2f} {row['error_day14']:>12.2f}")
    
    print(f"\nWorst behaved TLEs (fastest error growth):")
    print(f"{'Name':<30} {'Slope (km/day)':>15} {'Error day7':>12} {'Error day14':>12}")
    print("-" * 72)
    for _, row in worst.iterrows():
        print(f"{row['name']:<30} {row['slope_km_day']:>15.2f} "
              f"{row['error_day7']:>12.2f} {row['error_day14']:>12.2f}")
    
    print(f"\nFleet error growth statistics:")
    print(f"   Mean slope:    {slope_df['slope_km_day'].mean():.2f} km/day")
    print(f"   Median slope:  {slope_df['slope_km_day'].median():.2f} km/day")
    print(f"   Best:          {slope_df['slope_km_day'].min():.2f} km/day — {slope_df.iloc[0]['name']}")
    print(f"   Worst:         {slope_df['slope_km_day'].max():.2f} km/day — {slope_df.iloc[-1]['name']}")
    
    return slope_df

slope_df = analyze_per_satellite_error(error_df, '2023-054')


Best behaved TLEs (slowest error growth):
Name                            Slope (km/day)   Error day7  Error day14
------------------------------------------------------------------------
TBA - TO BE ASSIGNED                      6.96        48.62       210.05
TBA - TO BE ASSIGNED                      8.89         2.52        61.48
TBA - TO BE ASSIGNED                     19.70        96.92       958.98
TBA - TO BE ASSIGNED                     29.54         9.11         9.11
TBA - TO BE ASSIGNED                     32.04         5.92        78.04
TBA - TO BE ASSIGNED                     35.43         2.59        59.65
TBA - TO BE ASSIGNED                     44.67        24.88       166.58
TBA - TO BE ASSIGNED                     50.69        11.35       105.76
TBA - TO BE ASSIGNED                     51.96         8.37        90.50
TBA - TO BE ASSIGNED                     56.72        18.35        52.50

Worst behaved TLEs (fastest error growth):
Name                            Slope

In [81]:
def resolve_names(slope_df, hist_df):
    """Replace TBA names with confirmed names and add INTLDES."""
    name_map   = {}
    intldes_map = {}
    
    for norad in slope_df['norad']:
        sat_hist = hist_df[hist_df['norad'] == norad].sort_values('epoch')
        
        # Find first confirmed name
        confirmed = sat_hist[~sat_hist['name'].str.startswith('TBA') & 
                             ~sat_hist['name'].str.startswith('OBJECT')]
        if len(confirmed) > 0:
            name_map[norad] = confirmed.iloc[0]['name']
        else:
            name_map[norad] = sat_hist.iloc[-1]['name']
        
        # Parse INTLDES from TLE line1
        try:
            line1       = sat_hist.iloc[-1]['line1']
            intldes_raw = line1[9:17].strip()
            year        = intldes_raw[:2]
            num         = intldes_raw[2:5]
            piece       = intldes_raw[5:]
            full_year   = f"19{year}" if int(year) >= 57 else f"20{year}"
            intldes_map[norad] = f"{full_year}-{num}{piece}"
        except:
            intldes_map[norad] = 'N/A'
    
    slope_df['confirmed_name'] = slope_df['norad'].map(name_map)
    slope_df['intldes']        = slope_df['norad'].map(intldes_map)
    return slope_df

slope_df = resolve_names(slope_df, results_t7['hist_df'])

# Print with all identifiers
for label, group in [('Best behaved', slope_df.head(10)), 
                      ('Worst behaved', slope_df.tail(10))]:
    print(f"\n{label} TLEs:")
    print(f"{'NORAD':<8} {'INTLDES':<15} {'Name':<25} {'Slope':>10} {'Err day7':>10} {'Err day14':>10}")
    print("-" * 82)
    for _, row in group.iterrows():
        print(f"{row['norad']:<8} {row['intldes']:<15} {row['confirmed_name']:<25} "
              f"{row['slope_km_day']:>10.2f} {row['error_day7']:>10.2f} {row['error_day14']:>10.2f}")


Best behaved TLEs:
NORAD    INTLDES         Name                           Slope   Err day7  Err day14
----------------------------------------------------------------------------------
56206    2023-054AE      LEMUR 2 ROMEO-N-LEO             6.96      48.62     210.05
56205    2023-054AD      FACSAT-2                        8.89       2.52      61.48
56193    2023-054R       HAWK-7B                        19.70      96.92     958.98
56183    2023-054F       OBJECT F                       29.54       9.11       9.11
56210    2023-054AJ      CONNECTA T2.1                  32.04       5.92      78.04
56209    2023-054AH      GHGSAT-C7                      35.43       2.59      59.65
56187    2023-054K       LEMUR 2 SPACEGUS               44.67      24.88     166.58
56190    2023-054N       NUSAT-36 (ANNIE CANNON)        50.69      11.35     105.76
56203    2023-054AB      NUSAT-37 (JOAN CLARKE)         51.96       8.37      90.50
56196    2023-054U       VIGORIDE 6                     5

# Early Error Predictive Power Hypothesis

In [88]:
# Select one TLE per day per satellite — most recent update of that day
daily_sequences = {}

for norad in results_t7['cluster_norads']:
    sat_hist = results_t7['hist_df'][results_t7['hist_df']['norad'] == norad].copy()
    sat_hist['epoch'] = pd.to_datetime(sat_hist['epoch'], utc=True)
    sat_hist['date']  = sat_hist['epoch'].dt.date
    
    # Take last TLE of each day — most refined estimate
    daily = (sat_hist.sort_values('epoch')
                     .groupby('date')
                     .last()
                     .reset_index())
    
    if len(daily) < 5:
        continue
    
    daily['days_since_launch'] = [
        (pd.Timestamp(d) - pd.Timestamp('2023-04-15')).days 
        for d in daily['date']
    ]
    
    daily_sequences[norad] = {
        'name': sat_hist.iloc[-1]['name'],
        'daily_tles': daily[['date', 'days_since_launch', 
                              'epoch', 'line1', 'line2']].to_dict('records')
    }

print(f"Satellites with daily sequences: {len(daily_sequences)}")
for norad, data in list(daily_sequences.items())[:5]:
    print(f"   {norad} — {data['name']} — {len(data['daily_tles'])} days")

Satellites with daily sequences: 36
   56179 — OBJECT B — 56 days
   56180 — OBJECT C — 56 days
   56181 — KILICSAT — 57 days
   56182 — LEMUR 2 ONREFLECTION — 56 days
   56183 — OBJECT F — 56 days


In [89]:
# For each satellite, for each day n:
# Local error: propagate day n TLE to day n+1, normalize by 1 day
# Global error: propagate day n TLE to day n+k, decompose into RTN
# Focus on along-track (T) since it drives pass timing

def get_rtn_error(r_prop, r_actual, v_ref):
    """Decompose position error into RTN components."""
    dr    = np.array(r_prop) - np.array(r_actual)
    R_hat = np.array(v_ref) / np.linalg.norm(v_ref)  # along velocity
    
    # Wait — correct RTN:
    r_vec = np.array(r_actual)
    v_vec = np.array(v_ref)
    R_hat = r_vec / np.linalg.norm(r_vec)           # radial
    N_hat = np.cross(r_vec, v_vec)
    N_hat = N_hat / np.linalg.norm(N_hat)           # normal
    T_hat = np.cross(N_hat, R_hat)                  # transverse/along-track
    
    return {
        'total':       np.linalg.norm(dr),
        'radial':      abs(np.dot(dr, R_hat)),
        'along_track': abs(np.dot(dr, T_hat)),
        'cross_track': abs(np.dot(dr, N_hat))
    }

k_days     = [1, 2, 3, 7, 14]
daily_errors = []

for norad, data in daily_sequences.items():
    tles   = data['daily_tles']
    n_days = len(tles)
    
    for n in range(min(14, n_days - 1)):
        try:
            # Day n TLE
            xn      = tles[n]
            sat_n   = Satrec.twoline2rv(xn['line1'], xn['line2'])
            t_n     = Time(xn['epoch'])
            day_n   = xn['days_since_launch']
            
            # Local error — propagate to day n+1
            xn1     = tles[n + 1]
            sat_n1  = Satrec.twoline2rv(xn1['line1'], xn1['line2'])
            t_n1    = Time(xn1['epoch'])
            dt_days = xn1['days_since_launch'] - day_n
            
            if dt_days <= 0:
                continue
            
            en1, rn1, vn1 = sat_n1.sgp4(t_n1.jd1, t_n1.jd2)  # actual
            e0,  r0,  v0  = sat_n.sgp4(t_n1.jd1,  t_n1.jd2)  # propagated
            
            if en1 != 0 or e0 != 0:
                continue
            
            local_rtn = get_rtn_error(r0, rn1, vn1)
            
            # Normalize by time interval
            local_error_norm = local_rtn['total']       / dt_days
            local_T_norm     = local_rtn['along_track'] / dt_days
            local_R_norm     = local_rtn['radial']      / dt_days
            
            row = {
                'norad':            norad,
                'name':             data['name'],
                'n':                n,
                'day_n':            day_n,
                'dt_local':         dt_days,
                'local_error_norm': local_error_norm,
                'local_T_norm':     local_T_norm,
                'local_R_norm':     local_R_norm,
            }
            
            # Global errors at k days
            for k in k_days:
                target_day = day_n + k
                
                # Find TLE at target day
                candidates = [t for t in tles 
                              if abs(t['days_since_launch'] - target_day) <= 1]
                if not candidates:
                    row[f'global_total_day{k}']  = np.nan
                    row[f'global_T_day{k}']      = np.nan
                    row[f'global_R_day{k}']      = np.nan
                    continue
                
                xk     = min(candidates, 
                             key=lambda x: abs(x['days_since_launch'] - target_day))
                sat_k  = Satrec.twoline2rv(xk['line1'], xk['line2'])
                t_k    = Time(xk['epoch'])
                
                ek, rk, vk = sat_k.sgp4(t_k.jd1, t_k.jd2)
                e0k, r0k, v0k = sat_n.sgp4(t_k.jd1, t_k.jd2)
                
                if ek != 0 or e0k != 0:
                    row[f'global_total_day{k}']  = np.nan
                    row[f'global_T_day{k}']      = np.nan
                    row[f'global_R_day{k}']      = np.nan
                    continue
                
                global_rtn = get_rtn_error(r0k, rk, vk)
                row[f'global_total_day{k}'] = global_rtn['total']
                row[f'global_T_day{k}']     = global_rtn['along_track']
                row[f'global_R_day{k}']     = global_rtn['radial']
            
            daily_errors.append(row)
            
        except Exception as ex:
            continue

daily_err_df = pd.DataFrame(daily_errors)
print(f"Total observations: {len(daily_err_df)}")
print(f"Satellites:         {daily_err_df['norad'].nunique()}")
print(f"\nSample:")
print(daily_err_df[['norad', 'n', 'day_n', 'local_error_norm', 
                     'local_T_norm', 'global_total_day7', 
                     'global_T_day7']].head(10).to_string())

Total observations: 504
Satellites:         36

Sample:
   norad  n  day_n  local_error_norm  local_T_norm  global_total_day7  global_T_day7
0  56179  0      3          0.113762      0.076148           6.124003       6.095287
1  56179  1      5          0.690761      0.682236           1.075269       0.874372
2  56179  2      6          0.248972      0.228129          16.850597      16.839784
3  56179  3      7          1.157084      1.155594          50.632090      50.632076
4  56179  4      8          1.412897      1.412894          29.156662      29.151908
5  56179  5      9         12.312824     12.310441         508.541737     508.190063
6  56179  6     10          0.371781      0.351004          18.413391      18.410722
7  56179  7     11          0.329212      0.322323          13.165088      13.163778
8  56179  8     12          0.519336      0.509989          12.182178      12.182116
9  56179  9     13          2.283391      2.282803           3.273879       3.256910


In [90]:
# Cell 3 — Predictive power with normalized local error

print("Predictive Power Analysis — Normalized & RTN Decomposed")
print("="*65)

results_corr = []

for k in k_days:
    for component in ['total', 'T', 'R']:
        global_col = f'global_{component}_day{k}'
        if global_col not in daily_err_df.columns:
            continue
        
        # Linear correlation
        clean = daily_err_df[['local_error_norm', global_col]].dropna()
        clean = clean[(clean['local_error_norm'] > 0) & 
                      (clean[global_col] > 0)]
        
        if len(clean) < 10:
            continue
        
        r_lin  = clean['local_error_norm'].corr(clean[global_col])
        r2_lin = r_lin ** 2
        
        # Log correlation
        log_clean = np.log(clean + 1e-6)
        r_log     = log_clean['local_error_norm'].corr(log_clean[global_col])
        r2_log    = r_log ** 2
        
        results_corr.append({
            'k_days':     k,
            'component':  component,
            'n':          len(clean),
            'r_linear':   r_lin,
            'r2_linear':  r2_lin,
            'r_log':      r_log,
            'r2_log':     r2_log
        })
        
        print(f"\nLocal norm vs {component} error at day {k}:")
        print(f"   N:          {len(clean)}")
        print(f"   Linear r:   {r_lin:.4f}  (R²={r2_lin:.4f})")
        print(f"   Log r:      {r_log:.4f}  (R²={r2_log:.4f})")

corr_results_df = pd.DataFrame(results_corr)

Predictive Power Analysis — Normalized & RTN Decomposed

Local norm vs total error at day 1:
   N:          494
   Linear r:   1.0000  (R²=1.0000)
   Log r:      1.0000  (R²=1.0000)

Local norm vs T error at day 1:
   N:          494
   Linear r:   1.0000  (R²=1.0000)
   Log r:      0.9758  (R²=0.9522)

Local norm vs R error at day 1:
   N:          494
   Linear r:   0.9690  (R²=0.9389)
   Log r:      0.4892  (R²=0.2393)

Local norm vs total error at day 2:
   N:          504
   Linear r:   0.9274  (R²=0.8600)
   Log r:      0.8706  (R²=0.7579)

Local norm vs T error at day 2:
   N:          504
   Linear r:   0.9274  (R²=0.8600)
   Log r:      0.8428  (R²=0.7103)

Local norm vs R error at day 2:
   N:          504
   Linear r:   0.8935  (R²=0.7983)
   Log r:      0.4815  (R²=0.2318)

Local norm vs total error at day 3:
   N:          504
   Linear r:   0.8868  (R²=0.7864)
   Log r:      0.7761  (R²=0.6023)

Local norm vs T error at day 3:
   N:          504
   Linear r:   0.8866  (R²

In [91]:
# Cell 4 — Visualize predictive power decay and build quality score

# Plot 1 — R² decay over time
fig = go.Figure()

for component, color, name in [('total', 'cyan', 'Total Error'),
                                 ('T',     'orange', 'Along-Track'),
                                 ('R',     'red', 'Radial')]:
    comp_df = corr_results_df[corr_results_df['component'] == component]
    fig.add_trace(go.Scatter(
        x=comp_df['k_days'],
        y=comp_df['r2_linear'],
        mode='lines+markers',
        name=name,
        line=dict(color=color, width=2)
    ))

fig.add_hline(y=0.5, line_dash='dash', line_color='gray',
              annotation_text='R²=0.5 threshold',
              annotation_font_color='gray')

fig.update_layout(
    title='Predictive Power Decay — Local Error vs Global Error',
    xaxis_title='Days Ahead',
    yaxis_title='R² (explained variance)',
    yaxis=dict(range=[0, 1.05], gridcolor='gray'),
    xaxis=dict(gridcolor='gray'),
    paper_bgcolor='black', plot_bgcolor='black',
    font=dict(color='white'),
    legend=dict(bgcolor='black')
)
fig.show()

# Plot 2 — scatter local error vs day 7 total error
clean = daily_err_df[['norad', 'name', 'local_error_norm', 
                        'global_total_day7']].dropna()
clean = clean[(clean['local_error_norm'] > 0) & 
              (clean['global_total_day7'] > 0)]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=clean['local_error_norm'],
    y=clean['global_total_day7'],
    mode='markers',
    marker=dict(size=6, color='cyan', opacity=0.6),
    name='Observations'
))

# Trend line
coeffs = np.polyfit(clean['local_error_norm'], 
                     clean['global_total_day7'], 1)
x_line = np.linspace(clean['local_error_norm'].min(),
                      clean['local_error_norm'].max(), 100)
fig2.add_trace(go.Scatter(
    x=x_line,
    y=coeffs[0] * x_line + coeffs[1],
    mode='lines',
    line=dict(color='red', width=2, dash='dash'),
    name=f'Trend (r={0.784:.3f})'
))

fig2.update_layout(
    title='Local Error vs Day 7 Global Error',
    xaxis_title='Local Consecutive Error (km/day normalized)',
    yaxis_title='Day 7 Global Error (km)',
    paper_bgcolor='black', plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
    legend=dict(bgcolor='black')
)
fig2.show()

# Build candidate quality score
# Lower local error = better quality = higher score
sat_quality = (daily_err_df.groupby(['norad', 'name'])
                            ['local_error_norm']
                            .agg(['mean', 'median', 'std'])
                            .reset_index())

sat_quality.columns = ['norad', 'name', 'mean_local_err', 
                        'median_local_err', 'std_local_err']

# Normalize to 0-100 score (higher = better tracked)
max_err = sat_quality['mean_local_err'].max()
sat_quality['quality_score'] = (
    100 * (1 - sat_quality['mean_local_err'] / max_err)
)
sat_quality = sat_quality.sort_values('quality_score', ascending=False)

print(f"\nCandidate Quality Scores (higher = better tracked):")
print(f"{'NORAD':<8} {'Name':<30} {'Mean Err':>10} {'Quality':>10}")
print("-" * 62)
for _, row in sat_quality.iterrows():
    print(f"{row['norad']:<8} {row['name']:<30} "
          f"{row['mean_local_err']:>10.3f} {row['quality_score']:>10.1f}")


Candidate Quality Scores (higher = better tracked):
NORAD    Name                             Mean Err    Quality
--------------------------------------------------------------
56209    GHGSAT-C7                           1.266       99.1
56195    GHOST-1                             1.425       98.9
56197    GHOST-2                             1.448       98.9
56189    IT'S ABOUT TIME                     1.614       98.8
56210    CONNECTA T2.1                       1.631       98.8
56205    FACSAT-2                            1.670       98.8
56208    OBJECT AG                           1.711       98.7
56190    NUSAT-36 (ANNIE CANNON)             1.729       98.7
56203    NUSAT-37 (JOAN CLARKE)              1.797       98.7
56202    NUSAT-38 (MARIA AGNESI)             1.834       98.6
56199    TOMORROW-R1                         1.852       98.6
56183    OBJECT F                            1.932       98.6
56179    OBJECT B                            1.972       98.5
56184    SSS-2B 

In [92]:
# Percentile-based normalization — robust to outliers
p95 = sat_quality['mean_local_err'].quantile(0.95)

sat_quality['quality_score'] = (
    100 * (1 - (sat_quality['mean_local_err'].clip(upper=p95) / p95))
).round(1)

sat_quality = sat_quality.sort_values('quality_score', ascending=False)

print(f"Normalization cap (P95): {p95:.3f} km/day")
print(f"\nCandidate Quality Scores:")
print(f"{'NORAD':<8} {'Name':<30} {'Mean Err':>10} {'Quality':>10}")
print("-" * 62)
for _, row in sat_quality.iterrows():
    print(f"{row['norad']:<8} {row['name']:<30} "
          f"{row['mean_local_err']:>10.3f} {row['quality_score']:>10.1f}")

Normalization cap (P95): 10.993 km/day

Candidate Quality Scores:
NORAD    Name                             Mean Err    Quality
--------------------------------------------------------------
56209    GHGSAT-C7                           1.266       88.5
56195    GHOST-1                             1.425       87.0
56197    GHOST-2                             1.448       86.8
56189    IT'S ABOUT TIME                     1.614       85.3
56210    CONNECTA T2.1                       1.631       85.2
56205    FACSAT-2                            1.670       84.8
56208    OBJECT AG                           1.711       84.4
56190    NUSAT-36 (ANNIE CANNON)             1.729       84.3
56203    NUSAT-37 (JOAN CLARKE)              1.797       83.6
56202    NUSAT-38 (MARIA AGNESI)             1.834       83.3
56199    TOMORROW-R1                         1.852       83.1
56183    OBJECT F                            1.932       82.4
56179    OBJECT B                            1.972       82.1
561